# PID velocity tuning

Closed-loop velocity controller for the two rear motors. Setpoint in m/s, encoder feedback, PWM out at 100 Hz.

Gains were tuned by hand. Z-N closed loop didn't really work here: with Ki=Kd=0 the motor won't even start until Kp is high enough to beat static friction, so I never got a clean sustained oscillation to read Ku/Tu off. Went manual instead.

Setup:
- 660 ticks/rev (measured), wheel radius 0.0341 m
- control loop 100 Hz
- velocity feedback uses an EMA filter (alpha=0.2) computed once per loop, otherwise the encoder window gets stolen by the telemetry/comms tasks and the measurement goes to garbage

Final gains: Kp=1500, Ki=6000, Kd=30. Validated at 0.2 / 0.5 / 1.0 m/s.

Side note: measured top speed is ~1.37 m/s at full duty, datasheet says 0.29. Gear ratio on the motor is clearly not 1:90. Fixed the vehicle speed limits accordingly.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
cols = ['tag','t_ms','L_ticks','R_ticks','L_mps','R_mps',
        'yaw','pitch','roll','lax','lay','laz','gx','gy','gz','accu']
df = pd.read_csv('step_response.csv', header=None, names=cols)

df['t'] = (df['t_ms'] - df['t_ms'].iloc[0]) / 1000.0
df.head()

In [ ]:
setpoint = 0.5

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(df['t'], df['L_mps'], label='left')
ax.plot(df['t'], df['R_mps'], label='right')
ax.axhline(setpoint, color='k', ls='--', lw=1, label='setpoint')
ax.set_xlabel('time [s]')
ax.set_ylabel('wheel velocity [m/s]')
ax.set_title('Step response - closed-loop PID')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('step_response.png', dpi=150)
plt.show()

In [ ]:
# metrics off the left wheel
v = df['L_mps'].values
t = df['t'].values

t10 = t[np.argmax(v >= 0.1 * setpoint)]
t90 = t[np.argmax(v >= 0.9 * setpoint)]
rise = t90 - t10

peak = v.max()
overshoot = max(0.0, (peak - setpoint) / setpoint * 100)

band = 0.02 * setpoint
outside = np.abs(v - setpoint) > band
settle = t[np.where(outside)[0][-1]] if outside.any() else 0.0

ss_err = setpoint - v[-20:].mean()

print(f'rise time (10-90%): {rise*1000:.0f} ms')
print(f'overshoot:          {overshoot:.1f} %')
print(f'settling (2%):      {settle*1000:.0f} ms')
print(f'steady-state error: {ss_err*1000:.1f} mm/s')